In [0]:
# Imports
import sys
from pyspark.sql.functions import *

# Append the path to the repo
user_name = spark.sql("SELECT current_user()").collect()[0][0]
sys.path.insert(0, f"/Workspace/Users/{user_name}/EV-Charging-Demand-Optimisation/energy-forecasting")

from datetime import datetime, timezone
from src.data.collectors.weather import fetch_regional_weather


In [0]:
# Fetch weather data
from_dt = datetime(2024, 1, 1, tzinfo=timezone.utc)
to_dt   = datetime(2024, 12, 31, tzinfo=timezone.utc)

pdf = fetch_regional_weather(from_dt, to_dt)


In [0]:
# Write weather data to bronze table
sdf = spark.createDataFrame(pdf)

sdf.createOrReplaceTempView("weather_staging")
spark.sql("CREATE TABLE IF NOT EXISTS bronze_weather_regional USING DELTA AS SELECT * FROM weather_staging WHERE 1=0")  

# Here I am using a MERGE INTO statement to "upsert" the data into the bronze table, this will avoid duplicates
spark.sql("""
MERGE INTO bronze_weather_regional             
  USING weather_staging                                              
  ON bronze_weather_regional.region_id = weather_staging.region_id
  AND bronze_weather_regional.timestamp = weather_staging.timestamp
  WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
display(spark.table("bronze_weather_regional").limit(50))


In [0]:
display(spark.table("bronze_weather_regional").describe())